EDA for all final dataset 

In [1]:
import pandas as pd
import numpy as np

In [5]:
#load data 
df = pd.read_csv("../../data/final_datasets/feature_matrix_all_dimensions.csv", dtype={"plr_id": str})

#Missings per PLR - are values missing im many PLRs or are they concentrated on some PLRs? 
num_cols = df.select_dtypes(include=[np.number]).columns
nan_per_plr = df[num_cols].isna().sum(axis=1)

print("NaNs per PLR – Dispersion:")
print(nan_per_plr.value_counts().sort_index())

missing = (
    df.loc[nan_per_plr > 0, ["plr_id", "plr_name", "bez"]]
    .assign(n_nan=nan_per_plr[nan_per_plr > 0])
    .sort_values("n_nan", ascending=False)
)
print(f"\n{len(missing)} von {len(df)} PLR with at least one missing value:")
print(missing.to_string(index=False))



NaNs per PLR – Dispersion:
0     499
1      27
2       5
3       4
5       1
8       2
9       1
12      1
28      2
Name: count, dtype: int64

43 von 542 PLR with at least one missing value:
  plr_id                          plr_name                             bez  n_nan
06200418                           Landweg        06 - Steglitz-Zehlendorf     28
03400831                      Pankower Tor                     03 - Pankow     28
05300838                        Gartenfeld                    05 - Spandau     12
10100205 Gewerbegebiet Bitterfelder Straße        10 - Marzahn-Hellersdorf      9
03300515               Blankenburger Süden                     03 - Pankow      8
04400725            Güterbahnhof Grunewald 04 - Charlottenburg-Wilmersdorf      8
12200411               Schumacher-Quartier              12 - Reinickendorf      5
07200412                Schöneberger Linse       07 - Tempelhof-Schöneberg      3
08200728                        Ortolanweg                   08 - Neuk

In [7]:
#are the missings in one column?
mask1 = nan_per_plr == 1
einzel = df.loc[mask1, num_cols]
spalte_je_plr = einzel.isna().idxmax(axis=1)
print("which column is missing in the 1-NaN-PLR:")
print(spalte_je_plr.value_counts())

#Do the heavily affected PLR coincide with the non-residential flag?
heavy = df.loc[nan_per_plr >= 5, ["plr_id", "plr_name", "re_miet_dimension_anwendbar"]]
print("\nHeavily affected PLR vs. residential-applicability flag:")
print(heavy.to_string(index=False))


which column is missing in the 1-NaN-PLR:
soc_median_income_2023    25
re_miete_trend             2
Name: count, dtype: int64

Heavily affected PLR vs. residential-applicability flag:
  plr_id                          plr_name  re_miet_dimension_anwendbar
03300515               Blankenburger Süden                            0
03400831                      Pankower Tor                            0
04400725            Güterbahnhof Grunewald                            1
05300838                        Gartenfeld                            0
06200418                           Landweg                            0
10100205 Gewerbegebiet Bitterfelder Straße                            0
12200411               Schumacher-Quartier                            1


#Result of inspecting the NaNs

Missing values are highly concentrated, not dispersed: 499 of 542 PLR are complete, and only 43 carry any NaN. The most affected — Landweg and Pankower Tor (28 each), plus Gartenfeld, Gewerbegebiet Bitterfelder Straße, Blankenburger Süden, Güterbahnhof Grunewald, and Schumacher-Quartier — are development sites, former railway/industrial land, and barely-populated quarters that legitimately lack structure across all three dimensions. The long single-NaN tail is a column property, not a PLR one: 25 of 27 cases trace to soc_median_income_2023. 

The gaps are therefore structural, calling for imputation of the income column and consistent cross-dimension flagging of the non-residential sites — though two heavily affected quarters (Güterbahnhof Grunewald, Schumacher-Quartier) carry re_miet_dimension_anwendbar == 1, showing the real-estate flag alone does not capture them. 


Correlation Matrix across all three dimensions of gentrification 

In [9]:
#Drop helper columns and flags 
cols_to_use = []
for c in num_cols:
    is_flag = c.endswith(("_anwendbar", "_outlier", "_unsicher"))
    is_helper = c.startswith("com_has_")
    if not is_flag and not is_helper:
        cols_to_use.append(c)

#Compute the Spearman correlation matrix
corr = df[cols_to_use].corr(method="spearman")

#Walk through every unique pair of variables once and collect the results
results = []
for i in range(len(cols_to_use)):
    for j in range(i + 1, len(cols_to_use)):   # j > i avoids duplicates & self-pairs
        var_a = cols_to_use[i]
        var_b = cols_to_use[j]
        rho = corr.loc[var_a, var_b]

        dim_a = var_a.split("_")[0]   # "re", "soc", or "com"
        dim_b = var_b.split("_")[0]
        is_cross_dim = dim_a != dim_b

        results.append({
            "var_a": var_a,
            "var_b": var_b,
            "rho": rho,
            "abs_rho": abs(rho),
            "cross_dim": is_cross_dim,
        })

pairs = pd.DataFrame(results)

#Redundancy-suspect pairs: |rho| > 0.95
high = pairs[pairs["abs_rho"] > 0.95].sort_values("abs_rho", ascending=False)
print(f"Variable pairs with |rho| > 0.95: {len(high)}")
print(high[["var_a", "var_b", "rho", "cross_dim"]].to_string(index=False))

#Early warning: strong cross-dimension pairs (|rho| > 0.8)
cross_strong = pairs

Variable pairs with |rho| > 0.95: 0
Empty DataFrame
Columns: [var_a, var_b, rho, cross_dim]
Index: []


#Result 

No variable pairs exceed the redundancy threshold of |rho| > 0.95, and not a single cross-dimension pair even reaches |rho| > 0.8. The three dimensions therefore carry genuinely independent information: the inner-outer gradient does not collapse the variables onto a shared centrality axis strongly enough to produce pairwise redundancy.